In [1]:
import numpy as np
import pandas as pd
from yaml import safe_load
import os
from tqdm import tqdm

In [2]:
# This notebook generates features for ALL formats in one run.
# It will create: ipl_features.pkl, odi_features.pkl, t20_features.pkl, test_features.pkl

FORMAT_CONFIGS = {
    "IPL": {
        "data_dir": "data/ipl",
        "match_type": "T20",
        "overs": 20,
        "competition": "IPL",  # your IPL YAMLs use info.competition == "IPL"
        "features_out": "ipl_features.pkl",
    },
    "ODI": {
        "data_dir": "data/odis",
        "match_type": "ODI",
        "overs": 50,
        "competition": None,
        "features_out": "odi_features.pkl",
    },
    "T20": {
        "data_dir": "data/t20s",
        "match_type": "T20",
        "overs": 20,
        "competition": None,
        "features_out": "t20_features.pkl",
    },
    "Test": {
        "data_dir": "data/tests",
        "match_type": "Test",
        # Tests don't have a fixed innings length; we approximate to 90 overs (1 day)
        "overs": 90,
        "competition": None,
        "features_out": "test_features.pkl",
    },
}

def list_yaml_files(data_dir: str) -> list[str]:
    if not os.path.isdir(data_dir):
        return []
    files = []
    for name in os.listdir(data_dir):
        path = os.path.join(data_dir, name)
        if os.path.isfile(path) and name.lower().endswith(".yaml"):
            files.append(path)
    return sorted(files)

{k: len(list_yaml_files(v["data_dir"])) for k, v in FORMAT_CONFIGS.items()}

{'IPL': 1169, 'ODI': 3073, 'T20': 4801, 'Test': 888}

In [3]:
import pickle


def build_features_for_format(format_name: str, cfg: dict) -> pd.DataFrame | None:
    filenames = list_yaml_files(cfg["data_dir"])
    if not filenames:
        print(f"[{format_name}] Skipping: no YAML files under {cfg['data_dir']}")
        return None

    # Load match-level rows
    dfs = []
    counter = 1
    for file in tqdm(filenames, desc=f"[{format_name}] Loading YAML"):
        with open(file, "r", encoding="utf-8") as f:
            df = pd.json_normalize(safe_load(f))
            df["match_id"] = counter
            dfs.append(df)
            counter += 1
    matches_df = pd.concat(dfs, ignore_index=True)

    # Drop noisy columns safely
    matches_df.drop(
        columns=[
            "meta.data_version",
            "meta.created",
            "meta.revision",
            "info.outcome.bowl_out",
            "info.bowl_out",
            "info.supersubs.South Africa",
            "info.supersubs.New Zealand",
            "info.outcome.eliminator",
            "info.outcome.result",
            "info.outcome.method",
            "info.neutral_venue",
            "info.match_type_number",
            "info.outcome.by.runs",
            "info.outcome.by.wickets",
        ],
        inplace=True,
        errors="ignore",
    )

    # Filters
    if "info.match_type" in matches_df.columns and cfg.get("match_type"):
        matches_df = matches_df[matches_df["info.match_type"] == cfg["match_type"]]
    if "info.overs" in matches_df.columns and cfg.get("overs"):
        matches_df = matches_df[matches_df["info.overs"] == cfg["overs"]]
        matches_df = matches_df.drop(columns=["info.overs"])
    if "info.competition" in matches_df.columns and cfg.get("competition"):
        matches_df = matches_df[matches_df["info.competition"] == cfg["competition"]]

    # Gender filter if present (keeps male matches only)
    if "info.gender" in matches_df.columns:
        matches_df = matches_df[matches_df["info.gender"] == "male"].copy()
        matches_df = matches_df.drop(columns=["info.gender"])

    matches_df.reset_index(drop=True, inplace=True)
    if matches_df.empty:
        print(f"[{format_name}] Skipping: no matches left after filtering")
        return None

    required_cols = [
        "match_id",
        "innings",
        "info.city",
        "info.venue",
        "info.teams",
        "info.dates",
        "info.balls_per_over",
    ]
    existing = [c for c in required_cols if c in matches_df.columns]
    matches_df = matches_df[existing].copy()

    def get_player_dismissed(ball_val: dict) -> str:
        dismissal_sources = [ball_val.get("wicket"), ball_val.get("wickets")]
        for dismissal in dismissal_sources:
            if isinstance(dismissal, dict):
                return dismissal.get("player_out") or dismissal.get("player_dismissed") or "0"
            if isinstance(dismissal, list):
                for item in dismissal:
                    if isinstance(item, dict):
                        player = item.get("player_out") or item.get("player_dismissed")
                        if player:
                            return player
        return "0"

    # Build deliveries (1st innings)
    records = []
    for _, row in tqdm(matches_df.iterrows(), total=len(matches_df), desc=f"[{format_name}] Deliveries"):
        match_id = row["match_id"]
        city = row.get("info.city")
        venue = row.get("info.venue")
        teams = row.get("info.teams", [])

        innings_list = row["innings"]
        if not isinstance(innings_list, list) or not innings_list:
            continue

        first_innings = innings_list[0].get("1st innings") if isinstance(innings_list[0], dict) else None
        if not first_innings:
            continue

        batting_team = first_innings.get("team")
        deliveries = first_innings.get("deliveries", [])

        for delivery in deliveries:
            for ball_key, ball_val in delivery.items():
                records.append(
                    {
                        "match_id": match_id,
                        "teams": teams,
                        "batting_team": batting_team,
                        "ball": str(ball_key),
                        "batsman": ball_val.get("batsman"),
                        "bowler": ball_val.get("bowler"),
                        "runs": ball_val.get("runs", {}).get("total", 0),
                        "player_dismissed": get_player_dismissed(ball_val),
                        "city": city,
                        "venue": venue,
                    }
                )

    delivery_df = pd.DataFrame.from_records(records)
    if delivery_df.empty:
        print(f"[{format_name}] Skipping: no deliveries built")
        return None

    def bowling_team_fn(row: pd.Series) -> str:
        for t in row["teams"]:
            if t != row["batting_team"]:
                return t
        return row["batting_team"]

    delivery_df["bowling_team"] = delivery_df.apply(bowling_team_fn, axis=1)
    delivery_df.drop(columns=["teams"], inplace=True)

    mask_city_null = delivery_df["city"].isnull()
    delivery_df.loc[mask_city_null, "city"] = (
        delivery_df.loc[mask_city_null, "venue"].astype(str).str.split().str[0]
    )

    # Feature engineering
    total_balls = int(cfg["overs"] * 6)

    delivery_df["over"] = delivery_df["ball"].apply(lambda x: int(str(x).split(".")[0]))
    delivery_df["ball_no"] = delivery_df["ball"].apply(lambda x: int(str(x).split(".")[1]))
    delivery_df = delivery_df.sort_values(["match_id", "over", "ball_no"]).reset_index(drop=True)

    delivery_df["balls_bowled"] = delivery_df["over"] * 6 + delivery_df["ball_no"]
    delivery_df["balls_left"] = (total_balls - delivery_df["balls_bowled"]).clip(lower=0)

    delivery_df["current_score"] = delivery_df.groupby("match_id")["runs"].cumsum()

    delivery_df["player_dismissed_flag"] = delivery_df["player_dismissed"].apply(lambda x: 0 if x == "0" else 1).astype(int)
    delivery_df["total_dismissed"] = delivery_df.groupby("match_id")["player_dismissed_flag"].cumsum()
    delivery_df["wickets_left"] = 10 - delivery_df["total_dismissed"]

    delivery_df["crr"] = (delivery_df["current_score"] * 6 / delivery_df["balls_bowled"]).replace([np.inf, -np.inf], 0.0).fillna(0.0)

    match_totals = delivery_df.groupby("match_id")["runs"].sum().rename("innings_total")
    delivery_df = delivery_df.merge(match_totals, on="match_id", how="left")

    delivery_df["last_five"] = (
        delivery_df.groupby("match_id")["runs"]
        .rolling(window=30, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    final_features_df = delivery_df[
        [
            "batting_team",
            "bowling_team",
            "city",
            "current_score",
            "balls_left",
            "wickets_left",
            "crr",
            "last_five",
            "innings_total",
        ]
    ].copy()
    final_features_df.dropna(inplace=True)

    print(f"[{format_name}] Features rows: {len(final_features_df):,}")
    return final_features_df

In [4]:
# Generate and save features for all configured formats

outputs = {}

for format_name, cfg in FORMAT_CONFIGS.items():
    df = build_features_for_format(format_name, cfg)
    if df is None:
        continue
    out_path = cfg["features_out"]
    pickle.dump(df, open(out_path, "wb"))
    outputs[format_name] = {"features_out": out_path, "rows": int(len(df))}

outputs

[IPL] Loading YAML: 100%|█| 1169/1169 [03:15<00:00,  5.97
[IPL] Deliveries: 100%|█| 1169/1169 [00:00<00:00, 2754.35


[IPL] Features rows: 144,131


[ODI] Loading YAML: 100%|█| 3073/3073 [24:07<00:00,  2.12
[ODI] Deliveries: 100%|█| 2504/2504 [00:04<00:00, 541.92i


[ODI] Features rows: 721,704


[T20] Loading YAML: 100%|█| 4801/4801 [13:56<00:00,  5.74
[T20] Deliveries: 100%|█| 3038/3038 [00:05<00:00, 559.95i


[T20] Features rows: 367,252


[Test] Loading YAML: 100%|█| 888/888 [31:04<00:00,  2.10s
[Test] Deliveries: 100%|█| 866/866 [00:02<00:00, 395.43it


[Test] Features rows: 536,913


{'IPL': {'features_out': 'ipl_features.pkl', 'rows': 144131},
 'ODI': {'features_out': 'odi_features.pkl', 'rows': 721704},
 'T20': {'features_out': 't20_features.pkl', 'rows': 367252},
 'Test': {'features_out': 'test_features.pkl', 'rows': 536913}}

In [5]:
# (No further steps needed below this cell)
# If you still have older cells below, you can ignore them.
pass

In [6]:
# (moved into build_features_for_format)
pass

In [7]:
# Legacy cell (disabled). The notebook now generates all formats above.
pass

In [8]:
# Legacy cell (disabled).
pass

In [9]:
# Legacy cell (disabled).
pass

In [10]:
# Legacy cell (disabled).
pass